In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/accepted_2007_to_2018Q4.csv.gz', low_memory=False)

cols = ['funded_amnt', 'term', 'int_rate', 'grade', 'emp_length',
        'home_ownership', 'annual_inc', 'purpose', 'dti',
        'loan_status', 'total_pymnt', 'recoveries']
df = df[cols]

# Para LGD, solo nos interesan los préstamos que SÍ cayeron en default
default_df = df[df['loan_status'] == 'Charged Off'].copy()

print(f"Préstamos en default (Charged Off): {len(default_df):,}")
default_df.head()

Préstamos en default (Charged Off): 268,559


,funded_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,purpose,dti,loan_status,total_pymnt,recoveries
13,18000.0,60 months,19.48,E,7 years,RENT,150000.0,debt_consolidation,9.39,Charged Off,9452.74,1618.90
25,4225.0,36 months,14.85,C,5 years,RENT,35000.0,debt_consolidation,15.22,Charged Off,2558.87,368.37
30,16000.0,36 months,12.88,C,10+ years,MORTGAGE,65000.0,small_business,18.96,Charged Off,17396.63,0.00
31,24250.0,60 months,24.24,F,4 years,MORTGAGE,75000.0,debt_consolidation,20.84,Charged Off,4124.42,0.00
33,25000.0,60 months,13.99,C,9 years,MORTGAGE,79000.0,debt_consolidation,34.53,Charged Off,14490.92,0.00


In [2]:
# Total recuperado = lo pagado antes del default + lo recuperado después
default_df['total_recuperado'] = default_df['total_pymnt'] + default_df['recoveries']

# LGD = 1 - (% recuperado sobre el monto original)
default_df['lgd'] = 1 - (default_df['total_recuperado'] / default_df['funded_amnt'])

# Limitamos entre 0% y 100% (por si algún caso raro da un número fuera de ese rango)
default_df['lgd'] = default_df['lgd'].clip(lower=0, upper=1)

print(default_df['lgd'].describe())

count    268559.000000
mean          0.404352
std           0.265494
min           0.000000
25%           0.187532
50%           0.418215
75%           0.603788
max           1.000000
Name: lgd, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Preparamos las mismas features que usamos para PD
features = ['funded_amnt', 'term', 'int_rate', 'grade', 'emp_length',
            'home_ownership', 'annual_inc', 'purpose', 'dti']

X_lgd = default_df[features].copy()
y_lgd = default_df['lgd']

# Mismas conversiones que hicimos antes: term y emp_length a número, grade a orden, resto a dummies
X_lgd['term'] = X_lgd['term'].str.extract(r'(\d+)').astype(float)

emp_map = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
           '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9, '10+ years': 10}
X_lgd['emp_length'] = X_lgd['emp_length'].map(emp_map)

grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
X_lgd['grade'] = X_lgd['grade'].map(grade_map)

X_lgd = pd.get_dummies(X_lgd, columns=['home_ownership', 'purpose'], drop_first=True)

# Completamos nulos con la mediana (mismo criterio que en PD)
X_lgd['emp_length'] = X_lgd['emp_length'].fillna(X_lgd['emp_length'].median())
X_lgd['dti'] = X_lgd['dti'].fillna(X_lgd['dti'].median())

print("Listo, features preparadas:", X_lgd.shape)

Listo, features preparadas: (268559, 25)


In [4]:
# Dividimos train/test
X_train_lgd, X_test_lgd, y_train_lgd, y_test_lgd = train_test_split(
    X_lgd, y_lgd, test_size=0.2, random_state=42
)

# Entrenamos una regresión lineal (el equivalente "clásico" a la regresión logística, pero para números continuos)
modelo_lgd = LinearRegression()
modelo_lgd.fit(X_train_lgd, y_train_lgd)

# Predecimos sobre el test
pred_lgd = modelo_lgd.predict(X_test_lgd)

# Evaluamos
mae = mean_absolute_error(y_test_lgd, pred_lgd)
r2 = r2_score(y_test_lgd, pred_lgd)

print(f"MAE (Error Absoluto Medio): {mae:.4f}")
print(f"R²: {r2:.4f}")

MAE (Error Absoluto Medio): 0.2214
R²: 0.0251


In [5]:
lgd_por_grado = default_df.groupby('grade')['lgd'].mean().sort_values()
print(lgd_por_grado.round(3))

grade
B    0.395
E    0.401
A    0.403
F    0.404
C    0.407
D    0.408
G    0.435
Name: lgd, dtype: float64


In [6]:
# Guardamos la LGD promedio por grado como diccionario, para usarlo en el próximo paso
lgd_dict = lgd_por_grado.to_dict()
print(lgd_dict)

{'B': 0.39531235820904487, 'E': 0.4012774241521245, 'A': 0.40323229097779156, 'F': 0.40405533987687187, 'C': 0.40688161743179835, 'D': 0.4084585237865838, 'G': 0.43478162447255925}
